# 07: plate_z ゾーン正規化

plate_z（投球の縦方向ホームプレート通過位置）を、打者ごとのストライクゾーン（sz_top, sz_bot）で正規化した特徴量 `plate_z_norm` を追加する。

$$
\text{plate\_z\_norm} = \frac{\text{plate\_z} - \text{sz\_bot}}{\text{sz\_top} - \text{sz\_bot}}
$$

- `plate_z_norm = 0` → ストライクゾーン下端
- `plate_z_norm = 1` → ストライクゾーン上端
- 0〜1 の範囲外はボールゾーン

**データフロー**: `preprocess_07/` → `preprocess_08/` → `statcast-customized/data/`

In [ ]:
from pathlib import Path
import shutil

import pandas as pd
import numpy as np

SRC_DIR = Path("/workspace/datasets/statcast-customized-tmp/preprocess_07")
DST_DIR = Path("/workspace/datasets/statcast-customized-tmp/preprocess_08")
FINAL_DIR = Path("/workspace/datasets/statcast-customized/data")

DST_DIR.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(SRC_DIR.glob("*.parquet"))
print(f"Number of source files: {len(parquet_files)}")
print(f"Example: {parquet_files[0].name}")


## 正規化前の plate_z 確認

In [ ]:
# サンプルファイルで確認
df_sample = pd.read_parquet(parquet_files[0])
print("Columns:", list(df_sample.columns))
print()
for col in ["plate_z", "sz_top", "sz_bot"]:
    print(f"{col}: mean={df_sample[col].mean():.4f}, std={df_sample[col].std():.4f}, "
          f"null={df_sample[col].isna().sum()}")
print(f"\nsz range (sz_top - sz_bot): mean={( df_sample['sz_top'] - df_sample['sz_bot']).mean():.4f}")


## plate_z_norm を計算して全ファイルに追加

In [ ]:
for pf in parquet_files:
    df = pd.read_parquet(pf)
    
    sz_range = df["sz_top"] - df["sz_bot"]
    # sz_range が 0 のケースをガード（実データでは起こらないが安全策）
    sz_range = sz_range.replace(0, np.nan)
    
    df["plate_z_norm"] = (df["plate_z"] - df["sz_bot"]) / sz_range
    
    dst_path = DST_DIR / pf.name
    df.to_parquet(dst_path, index=False)

print(f"Processed {len(parquet_files)} files → {DST_DIR}")


## 結果検証

In [ ]:
# 処理結果の検証
df_check = pd.read_parquet(DST_DIR / parquet_files[0].name)
print("New columns:", list(df_check.columns))
print(f"\nplate_z_norm stats:")
print(df_check["plate_z_norm"].describe())
print(f"\nNull count: {df_check['plate_z_norm'].isna().sum()}")
print(f"\nExpected: 0.0=zone bottom, 1.0=zone top")
print(f"Rows in zone (0-1): {((df_check['plate_z_norm'] >= 0) & (df_check['plate_z_norm'] <= 1)).sum()} / {len(df_check)}")


## 最終データディレクトリにコピー

In [ ]:
# preprocess_08 → statcast-customized/data/ にコピー
for src_file in sorted(DST_DIR.glob("*.parquet")):
    dst_file = FINAL_DIR / src_file.name
    shutil.copy2(src_file, dst_file)

print(f"Copied {len(list(DST_DIR.glob('*.parquet')))} files → {FINAL_DIR}")
